# Data quality checks and missing values

*Session 4, block 1 · Course team · Licence: CC BY 4.0 (curated third-party parts keep their licence, named at each section)*

After this block you can check a table systematically and express every check as code: you name the dimension of data quality a problem belongs to, count the failing rows, and turn the rules into tests that stop a pipeline when new data break them. You can also ask *why* values are missing (MCAR, MAR, MNAR), decide whether to drop, fill or impute, and choose between simple, KNN and iterative imputation.

All examples use the Berlin listings of Inside Airbnb (snapshot of 26 June 2026, CC BY 4.0), downloaded as `airbnb-v1.zip` from the course drive into `case-study/data/` (see [case-study/README.md](../../../case-study/README.md)). The data come from public listing pages; we report them in aggregate and never try to identify a host.

## Overview

| Section | What it covers |
|---|---|
| Dimensions of data quality | completeness, validity, uniqueness, consistency, accuracy, timeliness, a first look at a table, `None`, `NaN` and `pd.NA` |
| Checks for types, ranges, duplicates and consistency | a check function with counts and severity, type checks, range and domain checks, placeholders, messy categories, fuzzy matching, impossible values, keys and near-duplicates, consistency within and across tables, accuracy and format of the registration field |
| Validation rules as tests | pytest tests and fixtures, expected failures, pandera schemas, class-based models, runtime checks, checkpoints, table-wide checks |
| Missing data mechanisms (MCAR, MAR, MNAR) | the three mechanisms, a hand example, a simulation, missing prices in the listings, the chi-square test, a missingness model, structural missingness |
| Dropping, filling and imputation | `dropna`, `fillna`, `ffill` and `bfill`, simple, KNN and iterative imputation compared, iterative imputer variants, missing indicators, leakage, single and multiple imputation, imputation before an estimator |
| Worked example | the data quality report: findings, severity and actions |
| Check your understanding, Exercises, Debrief | five questions, nine exercises in pairs, what to compare afterwards |

What happens to a new snapshot of the data, from the first checks to the cleaning of block 2:

```mermaid
flowchart LR
    D["New data"] --> C["Checks as code"]
    C --> R["Quality report<br/>finding, severity, action"]
    R --> F{"Critical<br/>finding?"}
    F -->|yes| S["Stop: fix the source<br/>or the loading script"]
    F -->|no| M["Missing values:<br/>why? drop, fill, impute"]
    M --> CL["Cleaning pipeline<br/>(block 2)"]
```

In [1]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

Next we load the prepared listings and look at their size. `pd.read_parquet` reads a Parquet file, a compressed column format that keeps the data types.

In [2]:
import numpy as np
import pandas as pd

listings = pd.read_parquet("case-study/data/airbnb/listings.parquet")
listings.shape

(10221, 42)

## Dimensions of data quality

**Data quality** means: how well do the data fit the question we want to answer? The same table can be good enough for one question and useless for another. To look for problems systematically instead of by luck, we use six **dimensions** (they appear, with small variations, in the standard ISO/IEC 25012 and in the quality frameworks of statistical offices):

| Dimension | Question | Everyday analogy |
|---|---|---|
| **Completeness** | Are the required values there? | a form with empty boxes |
| **Validity** | Right type, format and range? | a birth date of 31 February |
| **Uniqueness** | Is each real thing recorded once? | the same guest booked twice |
| **Consistency** | Do related values agree, within and across tables? | a check-out date before the check-in |
| **Accuracy** | Do values describe reality? | a correct-looking phone number that belongs to someone else |
| **Timeliness** | Are the data recent enough? | last year's timetable |

Naming the dimension tells you where to look and what to do. Without a list, checks depend on what the analyst happens to notice, and whole classes of problems (duplicates, contradictions between tables) go unseen.

### Six dimensions on five rows

Five invented rows of a listings table:

| id | room_type | price | minimum_nights | maximum_nights |
|---|---|---|---|---|
| 101 | Entire home/apt | $95.00 | 2 | 30 |
| 102 | Entire home/apt | | 3 | 1125 |
| 103 | private room | $40.00 | 1 | 2147483647 |
| 101 | Entire home/apt | $95.00 | 2 | 30 |
| 104 | Private room | $1,250.00 | 400 | 365 |

**Predict.** Which dimension does each problem in these five rows belong to? Write it down before you read on.

Completeness: one price is missing. Validity: prices are text with a currency sign, `private room` has the wrong case, and 2,147,483,647 is not a stay but the largest number a 32-bit integer can hold, a software default. Uniqueness: listing 101 appears twice. Consistency: listing 104 asks for at least 400 nights but allows at most 365.

### Two dimensions in one line each: completeness and timeliness

Two dimensions can be looked at in one line each. `isna().mean()` gives the share of missing values per column (completeness); the scrape dates tell us how old the data are (timeliness).

In [3]:
print(listings.isna().mean().sort_values(ascending=False).head(4).round(3).to_dict())
print(listings["last_scraped"].agg(["min", "max"]).dt.date.to_dict())

{'bathrooms': 0.524, 'beds': 0.492, 'price': 0.339, 'estimated_revenue_l365d': 0.339}
{'min': datetime.date(2026, 6, 26), 'max': datetime.date(2026, 7, 3)}


Half of the listings have no `bathrooms` or `beds`, and a third have no price. The snapshot was scraped over eight days; prices and availability change daily, so every number in this course describes the market at the end of June 2026. Accuracy is the hardest dimension to check from the table alone: it needs an outside reference (a site visit, an official register). The section on checks meets one accuracy problem that is visible in the data.

> **Note.** Missing values hide in many forms: `""`, `"n/a"`, `0` for an unknown price, `1900-01-01` for an unknown date, `-999` in survey data, `2147483647` for "no limit". `isna()` sees none of them. Look at the most frequent and the largest values of a column before you trust it.

### A first look at a table

*Adapted from [Data Preparation](https://github.com/microsoft/Data-Science-For-Beginners/blob/main/2-Working-With-Data/08-data-preparation/notebook.ipynb) (Microsoft, Data Science for Beginners, MIT).*

When a table has thousands of rows and dozens of columns, a few attributes and methods give a first overview before any check is written:

- `df.shape` is an *attribute* (no brackets): the number of rows and columns.
- `df.columns` lists the column names, and nothing else.
- `df.info()` shows, per column, the data type and the number of **non-null** values. A column whose type is `object` or `str` but should be a number is a type problem; a non-null count below the number of rows is a completeness problem.
- `df.describe()` summarises numeric columns: count, mean, standard deviation, minimum, quartiles and maximum. Suspicious minima and maxima are often the first sign of a placeholder.
- `df.head()` and `df.tail()` show the first and last rows; `head(10)` shows ten.

Next we run `info()` on a handful of columns. `df[list_of_columns]` selects columns by name.

In [4]:
listings[["id", "room_type", "accommodates", "bedrooms", "price", "minimum_nights", "maximum_nights", "first_review"]].info()

<class 'pandas.DataFrame'>
RangeIndex: 10221 entries, 0 to 10220
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id              10221 non-null  int64         
 1   room_type       10221 non-null  str           
 2   accommodates    10221 non-null  int64         
 3   bedrooms        7506 non-null   float64       
 4   price           6754 non-null   float64       
 5   minimum_nights  10219 non-null  float64       
 6   maximum_nights  10219 non-null  float64       
 7   first_review    8218 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(4), int64(2), str(1)
memory usage: 779.2 KB


**Predict.** Which of these columns will show a suspicious maximum in `describe()`? `.T` turns the summary on its side so that each column becomes a row.

In [5]:
listings[["accommodates", "bedrooms", "price", "minimum_nights", "maximum_nights"]].describe().T.round(1)

,count,mean,std,min,25%,50%,75%,max
accommodates,10221.0,3.1,2.0,1.0,2.0,2.0,4.0,1.600000e+01
bedrooms,7506.0,1.5,0.8,0.0,1.0,1.0,2.0,1.400000e+01
price,6754.0,161.0,219.8,2.7,70.1,130.0,205.5,1.002500e+04
minimum_nights,10219.0,34.6,52.5,1.0,1.0,3.0,92.0,1.125000e+03
maximum_nights,10219.0,420885.7,30041339.4,1.0,180.0,365.0,1125.0,2.147484e+09


The maximum stay of 2,147,483,647 nights and a minimum stay of 1,125 nights stand out immediately; a price of €10,025 a night is a question for block 2. The counts in the first column of `describe()` repeat what `info()` said about missing values.

### How pandas stores a missing value: `None`, `NaN` and `pd.NA`

*Code adapted from [Handling Missing Data](https://github.com/jakevdp/PythonDataScienceHandbook/blob/master/notebooks/03.04-Missing-Values.ipynb) (Jake VanderPlas, Python Data Science Handbook, code MIT); the explanation is our own.*

A table needs a marker for "no value here". There are two classic strategies: a separate **mask** that says which cells are missing, or a **sentinel**, a special value stored in the cell itself. pandas uses sentinels, and it has three of them:

- **`None`** is Python's "nothing". A NumPy array that contains `None` cannot keep a fast numeric type; it falls back to `dtype=object`, an array of general Python objects. Every operation then runs in slow Python code, and sums fail, because Python does not know how to add `None` to a number.
- **`NaN`** ("not a number") is a special floating-point value defined by the IEEE 754 standard. It keeps arrays fast, but it spreads: any arithmetic with `NaN` gives `NaN`. NumPy's ordinary aggregations therefore return `NaN`, while the `nan`-aware versions (`np.nansum`, `np.nanmin`, `np.nanmax`) skip it. There is no `NaN` for integers, strings or Booleans.
- **`pd.NA`** is pandas' own marker, used by the **nullable** types written with a capital letter (`Int64`, `Int32`, `boolean`, `string`). An integer column can then hold missing values without becoming a float column.

Next we see the cost of `None` in a NumPy array. **Predict** the `dtype` of the first array and what `.sum()` does. The error is intended; `try`/`except` catches it and prints the message.

In [6]:
vals1 = np.array([1, None, 2, 3])
print(vals1.dtype)
try:
    vals1.sum()
except TypeError as err:
    print("TypeError:", err)

object
TypeError: unsupported operand type(s) for +: 'int' and 'NoneType'


With `NaN` the array stays numeric, but every ordinary aggregate is "infected". The `nan`-functions ignore the missing value.

In [7]:
vals2 = np.array([1, np.nan, 3, 4])
print(vals2.dtype, 1 + np.nan, 0 * np.nan)
print(vals2.sum(), vals2.min(), vals2.max())
print(np.nansum(vals2), np.nanmin(vals2), np.nanmax(vals2))

float64 nan nan
nan nan nan
8.0 1.0 4.0


#### Upcasting and nullable types

pandas converts between the markers for you. **Predict** what happens to an integer Series when one value is set to `None`; then compare with the nullable type `Int32`, which keeps integers and shows `<NA>`.

In [8]:
x = pd.Series(range(2), dtype=int)
x[0] = None
print(x.dtype, x.tolist())
print(pd.Series([1, np.nan, 2, None, pd.NA], dtype="Int32").tolist())

float64 [nan, 1.0]
[1, <NA>, 2, <NA>, <NA>]


The integer Series became `float64` with `NaN`: pandas **upcasts** to a type that can hold the marker. In the listings, `accommodates` is a complete integer column, while `bedrooms` is stored as a float because it has gaps.

#### What `isnull()` counts as missing

Which values does `isnull()` treat as missing? **Predict** each of the four.

In [9]:
example = pd.Series([0, np.nan, "", None])
example.isnull().tolist()

[False, True, False, True]

`0` is a perfectly good number and `""` is a perfectly good (empty) string, so pandas does not count them as missing. That is exactly why placeholders slip through `isna()`: you must decide, column by column, which values mean "unknown".

> 📖 **Read more:** [extras/03-missing-values-in-pandas.ipynb](extras/03-missing-values-in-pandas.ipynb) (Python Data Science Handbook): the original text on masks and sentinels, with timing of object arrays. With pandas 3, write `ffill()` and `bfill()` where it uses `fillna(method=...)`.

## Checks for types, ranges, duplicates and consistency

A **check** is a rule that every row should satisfy, written so that a program can evaluate it. Four kinds cover most problems:

- **Type checks**: a price stored as text cannot be averaged.
- **Range and domain checks**: values lie in an allowed range (at least one guest, a minimum stay of at most a year) or come from an allowed set (four room types), and codes follow their format.
- **Uniqueness checks**: keys are unique, and the same real offer is not recorded twice under different ids.
- **Consistency checks**: rules that relate columns (minimum stay not above the maximum) or tables (every listing in the calendar exists in the listings table).

### Counting failures and rating severity

A good check returns a **count** of failing rows, not just "fail": the count tells you whether the problem is an exception or a pattern. Each finding also gets a **severity**: *critical* (a key or hard rule is broken; the data cannot be used as they are), *major* (many rows or a central variable; handle before analysis), *minor* (document it; handle it when it matters).

Next we write one small function that records a check as a row of a report and prints the count. A Boolean Series has `True` for every failing row, so `.sum()` counts them.

In [10]:
results = []


def check(dimension, column, rule, failed, severity):
    # Record one check; failed is a Boolean Series (True = the row fails the rule).
    n_failed = int(failed.sum())
    results.append({"dimension": dimension, "column": column, "rule": rule,
                    "n_failed": n_failed, "severity": severity if n_failed else "ok"})
    print(f"{rule}: {n_failed:,} failing")

### Type checks: a price stored as text

A **type check** tests that a column holds the data type the analysis needs. In the course file the price is already a number: the lecturer's preparation script converted it. In the original Inside Airbnb file `listings.csv.gz`, every price is text such as `$1,250.00`. The next cell rebuilds that text column from the prepared prices with an f-string (`:,.2f` writes a thousands separator and two decimals), so that we can watch the check fail and then fix it. `pd.to_numeric(..., errors="coerce")` turns every value that is not a number into `NaN`, so the failing rows are those that were present but did not convert.

In [11]:
# The original file stores prices as text such as "$1,250.00"; rebuild that format from the prepared prices.
raw_price = listings["price"].map(lambda p: f"${p:,.2f}", na_action="ignore")
print(raw_price.dropna().head(3).tolist())
as_number = pd.to_numeric(raw_price, errors="coerce")
check("validity", "price (original text)", "price is a number", as_number.isna() & raw_price.notna(), "critical")

['$160.71', '$193.33', '$10.88']
price is a number: 6,754 failing


Every price in the original file is text with a dollar sign (Inside Airbnb writes `$` for every city, although Berlin prices are in euros). The fix is a rule written as code: remove `$` and `,`, then convert. `str.replace` with `regex=True` removes every character in the set `[$,]`.

In [12]:
parsed = pd.to_numeric(raw_price.str.replace(r"[$,]", "", regex=True), errors="coerce")
print(parsed.notna().sum(), "of", raw_price.notna().sum(), "parsed")
print("same as the prepared table:", np.allclose(parsed.dropna(), listings["price"].dropna()))

6754 of 6754 parsed
same as the prepared table: True


### Range and domain checks

A **range check** tests that a number lies between two limits; a **domain check** tests that a value comes from an allowed set (the four room types) or area (Berlin's bounding box). `Series.between(a, b)` is `True` inside the range; `~` negates a Boolean Series. **Predict.** How many listings ask for a minimum stay of more than a year?

In [13]:
check("validity", "room_type", "one of four room types",
      ~listings["room_type"].isin(["Entire home/apt", "Private room", "Hotel room", "Shared room"]), "critical")
check("validity", "latitude, longitude", "inside Berlin's bounding box",
      ~listings["latitude"].between(52.33, 52.68) | ~listings["longitude"].between(13.08, 13.77), "critical")
check("validity", "accommodates", "at least one guest", listings["accommodates"].lt(1), "critical")
check("validity", "maximum_nights", "not the 32-bit placeholder 2,147,483,647",
      listings["maximum_nights"].eq(2**31 - 1), "minor")
check("validity", "minimum_nights", "minimum stay at most 365 nights", listings["minimum_nights"].gt(365), "minor")
print(listings.loc[listings["minimum_nights"].gt(365), "minimum_nights"].value_counts().sort_index().to_dict())

one of four room types: 0 failing
inside Berlin's bounding box: 0 failing
at least one guest: 0 failing
not the 32-bit placeholder 2,147,483,647: 2 failing
minimum stay at most 365 nights: 8 failing
{500.0: 1, 720.0: 1, 1000.0: 5, 1125.0: 1}


Eight listings require 500 to 1,125 nights. The platform allows it, but such listings are not holiday rentals.

#### Placeholders and default values

A **placeholder** or **default value** is a value that software fills in when nothing was entered. It looks valid but means "unknown" or "no limit", like the 2,147,483,647 nights checked above. The most frequent maximum stay is worth a look: `value_counts().head()` shows the most common values of a column.

In [14]:
listings["maximum_nights"].value_counts().head(3)

maximum_nights
1125.0    3822
365.0     2546
180.0      444
Name: count, dtype: int64

1,125 nights is the value Airbnb fills in when a host sets no maximum. It is valid, but it means "no limit chosen", not "a guest may stay three years". Never average `maximum_nights` without deciding what to do with such values.

### A dirty table: categories and impossible values

*Adapted from [Data Preparation](https://github.com/microsoft/Data-Science-For-Beginners/blob/main/2-Working-With-Data/08-data-preparation/notebook.ipynb) (Microsoft, Data Science for Beginners, MIT).* The fuzzy matching uses Python's built-in `difflib` instead of `rapidfuzz`, which is not in the course environment.

Range and domain checks catch values that break a rule we know in advance. Real data contain subtler problems:

1. **Inconsistent categories**: the same category spelled differently ("USA", "U.S.A", "United States").
2. **Impossible numbers**: values that domain knowledge rules out (an age of 199 or −5).
3. **Near-duplicates**: records of the same entity with small variations ("John Smith" and "John  Smith"), taken up under the uniqueness checks below.

The Berlin listings are already tidy in their categories, so we use the small "dirty" customer table of the source to see each problem. Next we create it.

In [15]:
dirty = pd.DataFrame({
    "customer_id": range(1, 13),
    "name": ["John Smith", "Jane Doe", "John Smith", "Bob Johnson", "Alice Williams", "Charlie Brown",
             "John  Smith", "Eva Martinez", "Bob Johnson", "Diana Prince", "Frank Castle", "Alice Williams"],
    "age": [25, 32, 25, 45, 28, 199, 25, 31, 45, 27, -5, 28],
    "country": ["USA", "UK", "U.S.A", "Canada", "USA", "United Kingdom",
                "United States", "Mexico", "canada", "USA", "UK", "usa"],
})
dirty["country"].value_counts().to_dict()

{'USA': 3,
 'UK': 2,
 'U.S.A': 1,
 'Canada': 1,
 'United Kingdom': 1,
 'United States': 1,
 'Mexico': 1,
 'canada': 1,
 'usa': 1}

#### Inconsistent categories

An **inconsistent category** is one category written in several ways. The table has seven spellings for four countries. A **mapping** from lower-cased spellings to one standard value fixes them. `str.lower()` lower-cases every value; `map(dict)` replaces each value by its entry in the dictionary and gives `NaN` for values that are not in it, which is useful: a new, unknown spelling then shows up as missing instead of slipping through.

In [16]:
country_mapping = {"usa": "USA", "u.s.a": "USA", "united states": "USA", "uk": "UK",
                   "united kingdom": "UK", "canada": "Canada", "mexico": "Mexico"}
dirty["country_clean"] = dirty["country"].str.lower().map(country_mapping)
dirty["country_clean"].value_counts().to_dict()

{'USA': 6, 'UK': 3, 'Canada': 2, 'Mexico': 1}

#### Fuzzy matching

When the spellings are not known in advance, **fuzzy matching** finds strings that are similar but not identical. `difflib.SequenceMatcher(None, a, b).ratio()` returns a similarity between 0 and 1. **Predict.** Will "USA" and "U.S.A" pass a threshold of 0.7?

In [17]:
from difflib import SequenceMatcher
from itertools import combinations

values = dirty["country"].unique()
for a, b in combinations(values, 2):
    score = SequenceMatcher(None, a.lower(), b.lower()).ratio()
    if score > 0.7:
        print(f"{a!r:18} ~ {b!r:18} {score:.2f}")

'USA'              ~ 'U.S.A'            0.75
'USA'              ~ 'usa'              1.00
'U.S.A'            ~ 'usa'              0.75
'Canada'           ~ 'canada'           1.00


Fuzzy matching finds the case variant and "USA"/"U.S.A", but not "USA"/"United States": the strings share too little. That is why a reviewed mapping beats a threshold for categories.

#### Impossible values

An **impossible value** is one that domain knowledge rules out, such as an age of 199. For impossible values, a domain rule comes first (an age is between 0 and 120), and only then a statistical rule (block 2). Here we set impossible ages to missing instead of deleting the customer. `Series.where(cond)` keeps values where the condition holds and puts `NaN` elsewhere.

In [18]:
dirty["age_clean"] = dirty["age"].where(dirty["age"].between(0, 120))
dirty.loc[dirty["age"] != dirty["age_clean"], ["customer_id", "age", "age_clean"]]

,customer_id,age,age_clean
5,6,199,NaN
10,11,-5,NaN


### Uniqueness checks: keys and repeated offers

A **uniqueness check** tests that each real thing is recorded once: a key column has no repeated value, and the same offer does not appear under two ids. `duplicated()` marks every row that repeats an earlier one; with a list of columns it compares only those. The key `id` must be unique. A subtler question: does the same host offer the same title, room type and number of guests more than once?

In [19]:
SAME_OFFER = ["host_id", "name", "room_type", "accommodates"]
check("uniqueness", "id", "id is unique", listings["id"].duplicated(), "critical")
check("uniqueness", ", ".join(SAME_OFFER), "same offer as another listing of the host",
      listings.duplicated(SAME_OFFER), "minor")
groups = listings[listings.duplicated(SAME_OFFER, keep=False)]
print(len(groups), "listings of", groups["host_id"].nunique(), "hosts are involved")

id is unique: 0 failing
same offer as another listing of the host: 226 failing
364 listings of 57 hosts are involved


No `id` repeats, so the key holds. The repeated offers belong to a few dozen hosts: operators of serviced apartments and hostels who offer several similar units under one title. They are real, separate offers, so we keep them; but a model validated with some units of a block in training and others in the test set will look better than it is (Session 7). Note the difference between `keep="first"` (the default: the first row of a group is not marked) and `keep=False` (every member of a group is marked).

#### Near-duplicates

*Adapted from [Data Preparation](https://github.com/microsoft/Data-Science-For-Beginners/blob/main/2-Working-With-Data/08-data-preparation/notebook.ipynb) (Microsoft, Data Science for Beginners, MIT).*

A **near-duplicate** is a record of the same entity with small variations, such as "John Smith" and "John  Smith" in the dirty table. Normalising first (strip spaces at the ends, collapse repeated spaces, lower-case) turns many near-duplicates into exact duplicates, which `duplicated(keep=False)` then shows as groups. `str.split().str.join(" ")` collapses repeated spaces.

In [20]:
dirty["name_norm"] = dirty["name"].str.split().str.join(" ").str.lower()
dirty[dirty.duplicated("name_norm", keep=False)].sort_values("name_norm")[["customer_id", "name", "age", "country_clean"]]

,customer_id,name,age,country_clean
4,5,Alice Williams,28,USA
11,12,Alice Williams,28,USA
3,4,Bob Johnson,45,Canada
8,9,Bob Johnson,45,Canada
0,1,John Smith,25,USA
2,3,John Smith,25,USA
6,7,John Smith,25,USA


Next we apply the same normalisation to the listing titles and count how many more "repeated offers" appear when case and spacing are ignored.

In [21]:
name_norm = listings["name"].str.split().str.join(" ").str.lower()
print(listings.duplicated(SAME_OFFER).sum(), "exact repeats;",
      listings.assign(name=name_norm).duplicated(SAME_OFFER).sum(), "after normalising the title")

226 exact repeats; 235 after normalising the title


#### What to do with duplicates

Once found, duplicates can be handled in four ways: keep the first occurrence (`drop_duplicates(keep="first")`), keep the last, **aggregate** the information of the group (for example the sum of purchases), or **flag** the group for human review. The choice depends on the question; record it (block 2).

> 📖 **Read more:** [extras/01-data-cleaning-missing-and-duplicates.ipynb](extras/01-data-cleaning-missing-and-duplicates.ipynb) (Microsoft, Data Science for Beginners): the original notebook with exercises on Iris, label and one-hot encoding, and a challenge row with several problems at once. The first `.sum()` error is intended.

### Consistency within a table and across tables

A **consistency check** tests that related values agree: columns of one row (the minimum stay not above the maximum) or rows of different tables. The calendar refers to listings by `listing_id`; every such id should exist in the listings table (a **foreign key**, Session 3). `isin` tests membership.

In [22]:
check("consistency", "minimum_nights", "minimum stay not above maximum stay",
      listings["minimum_nights"].gt(listings["maximum_nights"]), "critical")
check("consistency", "bedrooms", "not more bedrooms than guests",
      listings["bedrooms"].gt(listings["accommodates"]), "minor")
check("consistency", "review_scores_rating", "rating missing exactly when there is no review",
      listings["review_scores_rating"].isna().ne(listings["number_of_reviews"].eq(0)), "critical")
calendar_ids = pd.Series(pd.read_parquet("case-study/data/airbnb/calendar.parquet", columns=["listing_id"])["listing_id"].unique())
check("consistency", "calendar.listing_id", "listing exists in listings (foreign key)",
      ~calendar_ids.isin(listings["id"]), "major")

minimum stay not above maximum stay: 0 failing
not more bedrooms than guests: 31 failing
rating missing exactly when there is no review: 0 failing
listing exists in listings (foreign key): 79 failing


A listing without reviews never has a rating, and every reviewed listing has one, so that rule passes. The calendar, however, mentions listings that are not in the listings table: Inside Airbnb scrapes the files separately, and a listing can disappear in between. An inner join would drop these rows silently.

#### Reviews against listings

The course has three tables: `listings`, `calendar` and `reviews_monthly` (reviews per listing and month; the table `reviews` in the Session 3 database). Two more consistency rules relate them: the review count of a listing should equal the sum of its monthly counts, and every listing in the review table should exist in `listings`. `groupby(...).sum()` adds the monthly counts; `reindex(..., fill_value=0)` gives listings without any review month a total of 0.

In [23]:
reviews_monthly = pd.read_parquet("case-study/data/airbnb/reviews_monthly.parquet")
monthly_total = reviews_monthly.groupby("listing_id")["n_reviews"].sum()
total = listings.set_index("id")["number_of_reviews"]
check("consistency", "number_of_reviews", "equals the sum of the monthly counts",
      total.ne(monthly_total.reindex(total.index, fill_value=0)), "critical")
check("consistency", "reviews_monthly.listing_id", "listing exists in listings (foreign key)",
      ~pd.Series(reviews_monthly["listing_id"].unique()).isin(listings["id"]), "major")
print("review months after June 2026:", reviews_monthly["month"].gt("2026-06-01").sum())

equals the sum of the monthly counts: 0 failing
listing exists in listings (foreign key): 57 failing
review months after June 2026: 18


The review counts agree with the monthly table (both come from the same review file), but again some listings of the review table are not in `listings`. The review months after June 2026 are reviews written while the scrape was still running: July 2026 is an **incomplete month** and must not be read as a drop in demand (Session 12).

### Accuracy: names in the registration field

An **accuracy** problem is a value that looks valid but does not describe reality. Berlin requires a registration number for short-term rentals, and Airbnb asks hosts to type it into a free-text field. In the original file many hosts typed their own name or the name of their company. A name of a private person is **personal data** under the GDPR, and the analysis needs only the *type* of entry, so the preparation script replaced every entry that is not a registration number by a category in `license_status` (**data minimisation**). We count categories and never print the field.

In [24]:
status = listings["license_status"]
print(status.value_counts().to_dict())
check("accuracy", "license", "registration field holds the name of a private host",
      status.eq("private host name"), "major")
check("accuracy", "license", "registration field holds the name of a company", status.eq("legal entity name"), "minor")

{'registration number': 3600, 'missing': 3277, 'legal entity name': 1981, 'private host name': 1328, 'other': 35}
registration field holds the name of a private host: 1,328 failing
registration field holds the name of a company: 1,981 failing


> **Note.** A quality check must not itself spread personal data. Investigate free-text fields by counting patterns (`str.match`, `str.startswith`), and never copy raw values of such a field into a report, an issue or a chat with a language model.

### Format checks on registration numbers

A **format check** tests that a code follows its expected pattern. On the registration numbers that remain: an official Berlin number looks like `01/Z/RA/123456-24`. `str.fullmatch` tests a regular expression against the whole value: `\d{2}` is two digits, `[A-Z]` one capital letter.

In [25]:
numbers = listings.loc[status.eq("registration number"), "license"]
official = numbers.str.fullmatch(r"\d{2}/[A-Z]/[A-Z]{2}/\d{6}-\d{2}")
check("validity", "license", "registration number in the official format", ~official, "minor")

registration number in the official format: 26 failing


## Validation rules as tests

A check run once in a notebook protects nothing next quarter, when Inside Airbnb publishes a new snapshot. A **validation rule** is a check that must pass before the data are used; if it fails, the pipeline **stops**. Writing such rules as **tests** reuses the tools of Session 2:

- **pytest**: each rule is a function whose name starts with `test_` and that `assert`s a condition. A known, accepted problem is marked `@pytest.mark.xfail(strict=True, reason=...)`: it is documented, and with `strict=True` pytest complains if the problem ever disappears, so that the test can become a normal test. pytest exits with code 0 if all tests pass and 1 otherwise, which is what stops a pipeline or a CI job.
- **pandera**: the expected shape of a DataFrame (columns, types, checks per column) is declared once as a **schema**; `schema.validate(df, lazy=True)` collects every failure in a table.

Good rules are **specific** (one property each), **quantified** ("at least one guest", not "plausible") and **versioned** with the code, so that a change of a rule is visible in the history.

```mermaid
sequenceDiagram
    participant L as Loading script
    participant V as Validation (pytest / pandera)
    participant A as Analysis
    L->>V: new listings snapshot
    V->>V: run all rules
    alt all rules pass
        V-->>A: table released
    else a rule fails
        V-->>L: stop, report failing rows
    end
```

### Rules as pytest tests

Next we write a small test file. In a project it lives next to the code; here we write it to a temporary folder, so that the notebook leaves no files behind. `tempfile.mkdtemp()` creates such a folder; the test reads the data path from an environment variable.

In [26]:
import tempfile
import textwrap

test_dir = Path(tempfile.mkdtemp())
(test_dir / "test_listings.py").write_text(textwrap.dedent("""
    import os
    import pandas as pd
    import pytest

    @pytest.fixture(scope="module")
    def listings():
        return pd.read_parquet(os.environ["LISTINGS"])

    def test_id_is_unique(listings):
        assert listings["id"].is_unique

    @pytest.mark.xfail(strict=True, reason="known: 2 listings carry the placeholder 2,147,483,647")
    def test_no_placeholder_maximum_nights(listings):
        assert not listings["maximum_nights"].eq(2**31 - 1).any()

    def test_minimum_stay_at_most_one_year(listings):
        assert listings["minimum_nights"].dropna().le(365).all()
"""));

#### Fixtures

A **fixture** (`@pytest.fixture`) prepares something that several tests need, here the table; `scope="module"` loads it once for the whole file instead of once per test.

#### Running the tests as a pipeline would

**Predict.** Which of the three tests pass, and what will the exit code be? Next we run pytest as a separate process with `subprocess.run`, the way a pipeline or CI job would. `-q` keeps the output short, `--tb=line` shortens the report of a failure, and `-p no:cacheprovider` stops pytest from writing a cache folder.

In [27]:
import subprocess
import sys

env = {**os.environ, "LISTINGS": str(Path("case-study/data/airbnb/listings.parquet").resolve())}
run = subprocess.run([sys.executable, "-m", "pytest", "-q", "--tb=line", "--color=no", "-p", "no:cacheprovider", "test_listings.py"],
                     cwd=test_dir, env=env, capture_output=True, text=True)
print("\n".join(run.stdout.strip().splitlines()[-2:]))           # the last two lines: failures and summary
print("exit code:", run.returncode, "-> pipeline stops" if run.returncode else "-> data released")

FAILED test_listings.py::test_minimum_stay_at_most_one_year - assert np.False_
1 failed, 1 passed, 1 xfailed in 0.39s
exit code: 1 -> pipeline stops


The key test passes, the known placeholder is an expected failure (`x`), and the rule on minimum stays fails (`F`): exit code 1. A loading script that runs these tests first would stop here until someone decides what to do with the eight listings (block 2 flags them).

#### The course's full rule set

The full rule set of the course, including the privacy rule for the registration field and the known problems as `xfail`, is in [quality/test_listings_quality.py](quality/test_listings_quality.py). We run it the same way. `PYTHONDONTWRITEBYTECODE=1` stops Python from leaving a `__pycache__` folder next to it; the tests on the cleaned table are skipped until block 2 produces that table.

In [28]:
run = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider", "sessions/04-data-quality/workbooks/quality"],
                     env={**os.environ, "PYTHONDONTWRITEBYTECODE": "1"}, capture_output=True, text=True)
print(run.stdout.strip().splitlines()[-1], "| exit code", run.returncode)

15 passed, 6 skipped, 5 xfailed in 0.52s | exit code 0


### The same rules as a pandera schema

*Class-based models, runtime checks and checkpoints adapted from [Try pandera](https://github.com/unionai-oss/pandera/blob/main/docs/source/notebooks/try_pandera.ipynb) (pandera developers, MIT).* The grocery example of the source is replaced by the listings.

**pandera** is a Python library that checks pandas DataFrames against a **schema**, a declaration of the expected columns, their types and their checks. `pa.Column(dtype, checks, nullable=...)` describes one column; `pa.Check.in_range`, `isin`, `le` and `lt` are ready-made checks, and `error=` gives a check a readable name. `lazy=True` runs all checks before raising `SchemaErrors`, whose `failure_cases` table lists every failing value.

In [29]:
import pandera.pandas as pa

schema = pa.DataFrameSchema({
    "id": pa.Column(int, unique=True),
    "room_type": pa.Column(str, pa.Check.isin(["Entire home/apt", "Private room", "Hotel room", "Shared room"])),
    "accommodates": pa.Column(int, pa.Check.ge(1)),
    "price": pa.Column(float, pa.Check.in_range(10, 1000), nullable=True),
    "minimum_nights": pa.Column(float, pa.Check.le(365), nullable=True),
    "maximum_nights": pa.Column(float, pa.Check.lt(2**31 - 1, error="placeholder 2,147,483,647"), nullable=True),
})
try:
    schema.validate(listings, lazy=True)
except pa.errors.SchemaErrors as err:
    print(err.failure_cases.groupby(["column", "check"]).size().to_string())

column          check                     
maximum_nights  placeholder 2,147,483,647       2
minimum_nights  less_than_or_equal_to(365)      8
price           in_range(10, 1000)            109


In a pipeline you would not catch the exception: it stops the run, and the table of failure cases says why. The price rule (€10 to €1,000 a night) is a judgement call, not a law of nature; block 2 shows that most prices outside it belong to medium-term listings.

#### Class-based models

pandera has two equivalent ways to write a schema:

- the **object-based** `DataFrameSchema` used above, which can be changed at run time (`add_columns`, `remove_columns`, `update_column`);
- the **class-based** `DataFrameModel`, close in spirit to Python's `dataclasses` and `pydantic` (Session 16): each column is a class attribute with a type annotation `Series[type]` and a `pa.Field(...)` with its checks. It works with type checkers such as mypy.

`coerce=True` asks pandera to **parse** a column into the declared type before checking, for example text `"95.0"` into the float 95.0. Next we write a model for a few listing columns. `Field(ge=1)` means "greater than or equal to 1"; `isin=[...]` restricts the values.

In [30]:
from pandera.typing import DataFrame, Series


class Listings(pa.DataFrameModel):
    id: Series[int] = pa.Field(unique=True)
    room_type: Series[str] = pa.Field(isin=["Entire home/apt", "Private room", "Hotel room", "Shared room"])
    accommodates: Series[int] = pa.Field(ge=1)
    price: Series[float] = pa.Field(gt=0, nullable=True, coerce=True)


Listings.validate(listings[["id", "room_type", "accommodates", "price"]]).shape

(10221, 4)

#### Runtime checks on functions

The decorator `@pa.check_types` validates every argument annotated `DataFrame[Model]` each time the function is called, and the return value if it is annotated too. Next we write a function that adds a price per guest; the model for its output **inherits** from `Listings` and adds one column, so the schema documents what the function promises.

In [31]:
class ListingsPerGuest(Listings):
    price_per_guest: Series[float] = pa.Field(gt=0, nullable=True)


@pa.check_types(lazy=True)
def add_price_per_guest(df: DataFrame[Listings]) -> DataFrame[ListingsPerGuest]:
    return df.assign(price_per_guest=df["price"] / df["accommodates"])


add_price_per_guest(listings[["id", "room_type", "accommodates", "price"]]).head(3)

,id,room_type,accommodates,price,price_per_guest
0,3176,Entire home/apt,2,160.71,80.355000
1,9991,Entire home/apt,7,193.33,27.618571
2,14325,Entire home/apt,1,NaN,NaN


#### Schemas as checkpoints

A **checkpoint** is a schema checked at a fixed point in the code, for example on the input and output of a function. Checkpoints catch bugs in the code, not only in the data. **Predict** what happens when the function has a typo in the new column name. The error is intended and caught.

In [32]:
@pa.check_types(lazy=True)
def add_price_per_guest_buggy(df: DataFrame[Listings]) -> DataFrame[ListingsPerGuest]:
    return df.assign(price_per_gest=df["price"] / df["accommodates"])   # typo


try:
    add_price_per_guest_buggy(listings[["id", "room_type", "accommodates", "price"]])
except pa.errors.SchemaErrors as err:
    print(err.failure_cases[["schema_context", "column", "check"]].to_string(index=False))

 schema_context           column               check
DataFrameSchema ListingsPerGuest column_in_dataframe


The output schema reports that `price_per_guest` is missing from the result. Without the schema, the typo would have produced a table with a wrongly named column, and the error would have shown up much later.

#### Custom and table-wide checks

Some rules involve several columns. `pa.Check(function)` accepts any function that returns a Boolean Series; on the schema level (`checks=[...]`) it receives the whole DataFrame. Next we add the consistency rule "minimum stay not above maximum stay" and a rule that the share of missing prices stays below 40 %, a **quantified** completeness rule that would catch a broken scrape.

In [33]:
table_rules = pa.DataFrameSchema(checks=[
    pa.Check(lambda d: ~(d["minimum_nights"] > d["maximum_nights"]), error="minimum stay not above maximum stay"),
    pa.Check(lambda d: d["price"].isna().mean() < 0.40, error="at most 40 % of prices missing"),
])
table_rules.validate(listings).shape

(10221, 42)

Both rules pass on this snapshot. The 40 % limit is a choice: about a third of the prices are missing today, so a new snapshot with half the prices missing would stop the pipeline and make someone look.

#### Object-based or class-based?

The two APIs are equivalent; choose the class-based one for type checking and inheritance, the object-based one when the schema is built or changed by code.

> 📖 **Read more:** [extras/02-try-pandera.ipynb](extras/02-try-pandera.ipynb) (pandera documentation): the original tutorial, with links to statistical hypothesis checks, schema inference from data and synthetic test data.

## Missing data mechanisms (MCAR, MAR, MNAR)

How to handle a missing value depends on **why** it is missing. Rubin (1976) named three mechanisms:

- **MCAR, missing completely at random**: whether a value is missing has nothing to do with any variable, like a coin flip. The rows with a value are a random sample of all rows.
- **MAR, missing at random**: missingness depends only on variables we *observe*. Younger people skip the income question more often, and we know their age.
- **MNAR, missing not at random**: missingness depends on the missing value itself or on something unobserved. People with high incomes skip the income question.

| Mechanism | Drop incomplete rows? | What helps |
|---|---|---|
| MCAR | unbiased, just fewer rows | any reasonable method |
| MAR | biased | imputation or weighting that uses the observed variables |
| MNAR | biased | assumptions, more data, sensitivity analysis |

### A small example by hand

Five people, income missing for two:

| age | income |
|---|---|
| 25 | ? |
| 30 | ? |
| 45 | 4,000 |
| 50 | 4,200 |
| 60 | 4,500 |

The missing values belong to the youngest people. If income rises with age, the mean of the observed incomes (4,233) overestimates the true mean: dropping the incomplete rows is biased. Because age is observed, the pattern is consistent with MAR, and a model of income given age can correct it.

### A simulation of the three mechanisms

With real data we never know the truth. A simulation does: we generate incomes that rise with age, delete 30 % of them under each mechanism, and compare the mean of what is left with the true mean. `rng.random(n) < p` is `True` with probability `p`; `np.where(cond, a, b)` lets that probability depend on age or on income.

In [34]:
rng = np.random.default_rng(1)
n = 10_000
age = rng.normal(45, 12, n)
income = 1_000 + 60 * age + rng.normal(0, 500, n)            # monthly income rises with age
sim = pd.DataFrame({"age": age, "income": income})

mechanisms = {"MCAR": rng.random(n) < 0.3,                                   # a coin flip for every row
              "MAR": rng.random(n) < np.where(age < 40, 0.6, 0.1),           # depends on observed age
              "MNAR": rng.random(n) < np.where(income > 4000, 0.6, 0.1)}     # depends on income itself
print("true mean income", round(income.mean()))
pd.DataFrame({name: {"share missing": miss.mean(), "mean income observed": sim.loc[~miss, "income"].mean(),
                     "mean age observed": sim.loc[~miss, "age"].mean()} for name, miss in mechanisms.items()}).T.round(2)

true mean income 3686


,share missing,mean income observed,mean age observed
MCAR,0.30,3680.53,44.84
MAR,0.27,3877.45,47.94
MNAR,0.28,3458.62,42.36


Under MCAR the observed mean is close to the truth; only the sample is smaller. Under MAR it is biased upwards, and the observed ages show why (the young are missing). Under MNAR it is biased downwards (the rich are missing), and the observed ages show almost nothing: no observed column explains the gap, which is what makes MNAR hard.

### Missing prices in the listings

The data can never prove MCAR or MAR. They can show that MCAR is **implausible**: if the share of missing values differs between groups of an observed variable, missingness is not a coin flip.

A **missingness indicator** is `True` where the value is missing. Next we compare its mean, the share of missing prices, across bands of reviews in the last twelve months. `pd.cut` puts numbers into bands.

In [35]:
price_missing = listings["price"].isna()
recent = pd.cut(listings["number_of_reviews_ltm"], [-1, 0, 2, 10, 10_000], labels=["0", "1-2", "3-10", ">10"])
print(round(price_missing.mean(), 3))
price_missing.groupby(recent, observed=True).mean().round(3)

0.339


number_of_reviews_ltm
0       0.589
1-2     0.228
3-10    0.162
>10     0.069
Name: price, dtype: float64

A third of the listings show no price, and listings without a recent review lack one about eight times as often as busy ones. So missing prices are **not MCAR**. But do reviews "cause" missing prices? **Predict** a third variable before running the next cell. Hint: Inside Airbnb reads the price from the listing page for a bookable date.

In [36]:
blocked = listings["availability_365"].eq(0)          # no free night in the next 365 days
print(price_missing.groupby(blocked).mean().round(3).to_dict())
bookable = ~blocked
print(price_missing[bookable].groupby(listings.loc[bookable, "number_of_reviews_ltm"].eq(0)).mean().round(3).to_dict())

{False: 0.097, True: 1.0}
{False: 0.046, True: 0.204}


![Share of listings without a price by recent reviews and availability, and the structural missingness of the review score and the price](../figures/missingness-pattern.png)

**What do you see?** Find the group of listings in which the share of missing prices is highest, and what the right panel shows for the review score.

A listing with **no free night** in the next year shows no price at all (2,744 of 2,745 are missing), and such listings also collect few reviews. That explains most of the pattern. Among bookable listings a gap remains (20 % without a price when there was no recent review, 5 % otherwise): dormant listings often show no price.

What this means:

- Availability and recent reviews are observed, so **MAR given these variables** is a reasonable working assumption. Whether a dormant host would charge more or less than others (MNAR) cannot be decided from these data.
- For blocked listings the price is **structurally missing**: there is no price to recover, just as a listing without reviews has no rating (right panel). Do not impute such values; analyse "the listings with a current price" and say so.

### A test and a model for missingness

Two tools turn "the shares look different" into numbers.

#### The chi-square test of independence

The **chi-square test of independence** asks whether the missing share is the same in all review bands; with about 10,000 listings its p-value is tiny for any visible difference, so look at the size of the differences too (the test is taught in Session 5; here we only read its result). `pd.crosstab` counts combinations; `scipy.stats.chi2_contingency` runs the test.

In [37]:
from scipy.stats import chi2_contingency

table = pd.crosstab(recent, price_missing)
chi2_stat, p_value, dof, _ = chi2_contingency(table)
print(table)
print(f"chi2 = {chi2_stat:.0f}, dof = {dof}, p = {p_value:.1e}")

price                  False  True 
number_of_reviews_ltm              
0                       1924   2754
1-2                      837    247
3-10                    1419    275
>10                     2574    191
chi2 = 2496, dof = 3, p = 0.0e+00


#### A missingness model

A **missingness model** goes further: if a classifier predicts the indicator better than chance (an area under the ROC curve, **AUC**, above 0.5), the data are not MCAR, and the columns it uses belong in any imputation model. Logistic regression, pipelines and the AUC are taught in Sessions 6 and 8; here we only read the number. `cross_val_score` fits the model five times on different parts of the data (Session 7).

In [38]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

features = pd.DataFrame({"blocked": blocked.astype(int),
                         "log_reviews_ltm": np.log1p(listings["number_of_reviews_ltm"]),
                         "accommodates": listings["accommodates"]})
for cols in (["log_reviews_ltm"], ["blocked"], list(features.columns)):
    model = make_pipeline(StandardScaler(), LogisticRegression())
    auc = cross_val_score(model, features[cols], price_missing, cv=5, scoring="roc_auc").mean()
    print(f"AUC {auc:.2f} using {cols}")

AUC 0.78 using ['log_reviews_ltm']
AUC 0.90 using ['blocked']
AUC 0.94 using ['blocked', 'log_reviews_ltm', 'accommodates']


Recent reviews alone predict a missing price well, availability better, and both together best: missingness is strongly related to observed columns.

### Structural missingness: the review score

The review score is the extreme case of missingness tied to an observed column: it is missing exactly for the listings without any review. Like the price of a blocked listing, it is **structurally missing** (no review, no rating).

In [39]:
no_reviews = listings["number_of_reviews"].eq(0)
print(pd.crosstab(no_reviews.rename("no reviews"), listings["review_scores_rating"].isna().rename("rating missing")))
print(listings.groupby(no_reviews)[["availability_365", "minimum_nights"]].median())

rating missing  False  True 
no reviews                  
False            8218      0
True                0   2003
                   availability_365  minimum_nights
number_of_reviews                                  
False                         114.0             2.0
True                          214.0            92.0


Listings without reviews have more free nights and much longer minimum stays: many are new or medium-term offers. Keep the rating missing and add the indicator `no_reviews`; "no reviews yet" is information in itself.

> **Note.** "The data are missing at random" is an assumption, not a finding. State it, and for important conclusions check how results change under a plausible MNAR scenario.

## Dropping, filling and imputation

*Dropping and filling adapted from [Data Preparation](https://github.com/microsoft/Data-Science-For-Beginners/blob/main/2-Working-With-Data/08-data-preparation/notebook.ipynb) (Microsoft, Data Science for Beginners, MIT); code examples also follow the Python Data Science Handbook (code MIT).*

There are three ways to deal with missing data: **drop** the rows (or columns) that contain them, **fill** them with one value, or **impute** them, estimating each value from the other columns. Dropping reduces the data; it is acceptable when the data are large, the share missing is small and the values are MCAR. A column that is almost empty may be dropped as a whole.

### Dropping rows and columns: `dropna`

`dropna()` drops every row with any missing value. **Predict** the result for the small table below; then see `axis="columns"`, `how="all"` and `thresh`.

In [40]:
df = pd.DataFrame([[1, np.nan, 2],
                   [2, 3, 5],
                   [np.nan, 4, 6]])
df[3] = np.nan
df

,0,1,2,3
0,1.0,NaN,2,NaN
1,2.0,3.0,5,NaN
2,NaN,4.0,6,NaN


- `dropna()` (default `how="any"`) drops rows with at least one missing value: here every row.
- `dropna(axis="columns", how="all")` drops only columns that are completely empty.
- `dropna(thresh=3)` keeps rows with at least three **non-missing** values.

In [41]:
print(df.dropna().shape)
print(df.dropna(axis="columns", how="all").columns.tolist())
print(df.dropna(axis="rows", thresh=3).index.tolist())

(0, 4)
[0, 1, 2]
[1]


`thresh` is handy for real tables. **Predict** how many listings keep at least 40 of the 42 columns.

In [42]:
print(len(listings), "listings;", len(listings.dropna()), "complete;",
      len(listings.dropna(thresh=40)), "with at least 40 values")

10221 listings; 3455 complete; 5075 with at least 40 values


Only about a third of the listings are complete in every column, so `dropna()` on the whole table would throw away two thirds of the data, and not at random. Drop on the columns the analysis needs (`dropna(subset=[...])`), and say how many rows that removes.

### Filling with one value: `fillna`, `ffill` and `bfill`

`fillna(value)` replaces missing values with one value; `fillna(df.mean())` fills each column with its own mean (a column that is entirely empty stays empty). For **categories** the most frequent value (the **mode**) is common; for **skewed numbers** the median is safer than the mean, because the median is robust to extreme values. `ffill()` (forward fill) copies the last valid value forward, `bfill()` (backward fill) the next valid value backward; they suit ordered data such as a time series of daily prices, not unordered tables. With `axis=1` they fill along a row.

In [43]:
s = pd.Series([1, np.nan, 2, None, 3], index=list("abcde"), dtype="Int32")
print(s.fillna(0).tolist(), s.ffill().tolist(), s.bfill().tolist())
print(df.ffill(axis=1).to_numpy().tolist())
print(df.fillna(df.mean()).to_numpy().round(1).tolist())

[1, 0, 2, 0, 3] [1, 1, 2, 2, 3] [1, 2, 2, 3, 3]
[[1.0, 1.0, 2.0, 2.0], [2.0, 3.0, 5.0, 5.0], [nan, 4.0, 6.0, 6.0]]
[[1.0, 3.5, 2.0, nan], [2.0, 3.0, 5.0, nan], [1.5, 4.0, 6.0, nan]]


In the listings, a natural ordered example is the calendar: if a date were missing a price, `ffill()` within a listing would carry the last known price forward. In unordered data, `ffill()` copies a value from whatever row happens to come before: never use it there.

### Imputation: simple, KNN and iterative

**Imputation** fills missing values with estimates, so that methods that need complete data can run (most scikit-learn models refuse `NaN`). Three families:

- **`SimpleImputer`** fills a column with one number: `strategy="mean"`, `"median"`, `"most_frequent"` or `"constant"` (with `fill_value`). Fast, but it ignores the other columns and shrinks the spread, because every filled row gets the same value.
- **`KNNImputer`** fills a value with the average of the *k* most similar rows (**nearest neighbours**), measured on the observed columns. Like asking the ten most similar flats how many bedrooms they have. Columns must be on comparable scales.
- **`IterativeImputer`** predicts each incomplete column from the others with a regression and repeats until the estimates settle. This is the idea of **MICE** (multiple imputation by chained equations).

```mermaid
flowchart LR
    S["SimpleImputer<br/>mean / median / constant"] -->|"uses other columns? no"| R1["fast; biased under MAR;<br/>spread shrinks"]
    K["KNNImputer<br/>average of k neighbours"] -->|"yes"| R2["local; needs scaling;<br/>slow on large data"]
    I["IterativeImputer<br/>regression per column"] -->|"yes"| R3["models relations;<br/>basis of MICE"]
```

#### Comparing imputers on hidden values

To compare imputers fairly we need the truth. So we take the entire homes **with** a known number of bedrooms, hide 30 % of the values on purpose, impute them, and compare with what we hid. Next we build that table; `rng.random(n) < 0.3` hides each row with probability 0.3, which makes the hidden values MCAR by construction.

In [44]:
homes = listings[listings["room_type"].eq("Entire home/apt") & listings["bedrooms"].notna()]
km = 111.2 * np.hypot(homes["latitude"] - 52.5219, (homes["longitude"] - 13.4132) * np.cos(np.radians(52.52)))
X = pd.DataFrame({"accommodates": homes["accommodates"], "bedrooms": homes["bedrooms"], "km_to_centre": km})
print(len(X), "entire homes;", "correlation guests-bedrooms", round(X["accommodates"].corr(X["bedrooms"]), 2))

rng = np.random.default_rng(0)
hide = rng.random(len(X)) < 0.3
truth = X.loc[hide, "bedrooms"].to_numpy()
X_missing = X.copy()
X_missing.loc[hide, "bedrooms"] = np.nan

6479 entire homes; correlation guests-bedrooms 0.75


The distance to Alexanderplatz uses a flat-earth approximation (one degree of latitude is about 111.2 km), accurate enough within a city.

**Predict.** Which imputer will make the smallest error, and how much better than the mean? Next we fit each imputer with `fit_transform` and measure the error on the hidden values with the **RMSE** (root mean squared error, in bedrooms). KNN gets standardised columns, because a distance in kilometres and a count of guests are not on the same scale. `IterativeImputer` must be enabled with an extra import because scikit-learn still calls it experimental.

In [45]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer

def rmse(filled):
    return round(float(np.sqrt(np.mean((np.asarray(filled) - truth) ** 2))), 3)

scaler = StandardScaler().fit(X_missing)
knn = scaler.inverse_transform(KNNImputer(n_neighbors=10).fit_transform(scaler.transform(X_missing)))
errors = {"mean": rmse(SimpleImputer(strategy="mean").fit_transform(X_missing)[hide, 1]),
          "median": rmse(SimpleImputer(strategy="median").fit_transform(X_missing)[hide, 1]),
          "KNN (k = 10)": rmse(knn[hide, 1]),
          "iterative": rmse(IterativeImputer(random_state=0).fit_transform(X_missing)[hide, 1]),
          "rule: median for the same number of guests":
              rmse(X_missing.groupby("accommodates")["bedrooms"].transform("median")[hide])}
pd.Series(errors, name="RMSE (bedrooms)")

mean                                          0.903
median                                        1.063
KNN (k = 10)                                  0.619
iterative                                     0.601
rule: median for the same number of guests    0.659
Name: RMSE (bedrooms), dtype: float64

The mean and median ignore the size of the flat. The methods that use the number of guests cut the error by about a third, because guests and bedrooms are strongly correlated; the distance to the centre adds little. A one-line domain rule (the median number of bedrooms of homes for the same number of guests) is about as good as the iterative imputer and easier to explain. Imputation can only move information from related columns; it cannot create information that is not there.

#### Variants of the iterative imputer

*Adapted from [Imputing missing values with variants of IterativeImputer](https://scikit-learn.org/stable/auto_examples/impute/plot_iterative_imputer_variants_comparison.html) (scikit-learn developers, BSD-3-Clause).* The California housing data of the source are replaced by the hidden bedroom counts from above.

`IterativeImputer` can use any regression model to predict a column from the others: `estimator=...`. The source compares a regularised linear model (`BayesianRidge`, the default), a random forest (which imitates the popular R package **missForest**), and a k-nearest-neighbours regressor. Note that `KNeighborsRegressor` inside `IterativeImputer` is not the same as `KNNImputer`: the latter measures distances while ignoring missing coordinates, the former is a model fitted on imputed data. `max_iter` and `tol` control how long the round-robin regression runs.

In [46]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import BayesianRidge
from sklearn.neighbors import KNeighborsRegressor

variants = {"BayesianRidge (default)": BayesianRidge(),
            "random forest (missForest-like)": RandomForestRegressor(n_estimators=30, max_depth=8, random_state=0, n_jobs=-1),
            "k-NN regressor": KNeighborsRegressor(n_neighbors=10)}
pd.Series({name: rmse(IterativeImputer(estimator=est, random_state=0, max_iter=20, tol=1e-2)
                      .fit_transform(X_missing)[hide, 1]) for name, est in variants.items()}, name="RMSE (bedrooms)")

BayesianRidge (default)            0.601
random forest (missForest-like)    0.621
k-NN regressor                     0.615
Name: RMSE (bedrooms), dtype: float64

With only two predictors, the variants end up close to each other and to the simple guests rule. Flexible estimators pay off when relations are non-linear and there are many columns; they cost time and can overfit.

> 📖 **Read more:** [extras/05-iterative-imputer-variants.ipynb](extras/05-iterative-imputer-variants.ipynb) (scikit-learn gallery): the original comparison, including a Nystroem kernel with ridge regression.

### Keeping track of what was imputed

A **missing indicator** is a 0/1 column that is 1 where a value was missing. `add_indicator=True` appends one per column with gaps that is 1 where the value was missing, so that a model can still use the fact that it was missing.

In [47]:
imp = SimpleImputer(strategy="median", add_indicator=True).fit(listings[["bedrooms", "review_scores_rating"]])
imp.get_feature_names_out().tolist()

['bedrooms',
 'review_scores_rating',
 'missingindicator_bedrooms',
 'missingindicator_review_scores_rating']

> **Note.** Fit an imputer on the training data only, then apply it to the test data; fitting it on all rows lets the test rows influence the filled values (Session 7 calls this **leakage**). Never impute a value that is missing by definition, such as the rating of a listing without reviews; use an indicator instead.

### Single and multiple imputation

**Single** imputation, as here, treats filled values as if they were measured and understates uncertainty. **Multiple imputation** creates several completed datasets with random variation, analyses each and combines the results with Rubin's rules; it is the standard for statistical inference (`statsmodels.imputation.mice`).

### Imputation before an estimator

*Adapted from [Imputing missing values before building an estimator](https://scikit-learn.org/stable/auto_examples/impute/plot_missing_values.html) (scikit-learn developers, BSD-3-Clause).* The diabetes and California housing data of the source are replaced by the listings.

The RMSE on hidden values asks "how close is the filled value?". In machine learning the question is often different: "how good is the **model** that uses the filled data?". The scikit-learn example answers it by putting each imputer in a **pipeline** before a random forest and comparing the cross-validated error. We do the same with the real gaps of the listings: predict the log price of short-stay listings from size and rating columns that have missing values of their own. A **pipeline** (`make_pipeline(imputer, model)`) fits the imputer on the training folds only, which avoids the leakage described above.

In [48]:
short = listings[listings["price"].notna() & listings["minimum_nights"].lt(28)]
X_ml = short[["accommodates", "bedrooms", "beds", "bathrooms", "review_scores_rating", "number_of_reviews"]]
y_ml = np.log(short["price"])
print(len(X_ml), "short-stay listings;", X_ml.isna().mean().round(2).to_dict())

5302 short-stay listings; {'accommodates': 0.0, 'bedrooms': 0.21, 'beds': 0.07, 'bathrooms': 0.13, 'review_scores_rating': 0.07, 'number_of_reviews': 0.0}


Each imputer gets `add_indicator=True`, as in the source. `scoring="neg_mean_squared_error"` returns the negative MSE (scikit-learn always maximises a score); we flip the sign. The forest is kept small (`n_estimators=50`) so that the cell runs in seconds. **Predict.** Will the imputer matter much for a random forest?

In [49]:
from sklearn.ensemble import RandomForestRegressor

def cv_mse(imputer):
    model = make_pipeline(imputer, RandomForestRegressor(n_estimators=50, min_samples_leaf=5, random_state=0, n_jobs=-1))
    return -cross_val_score(model, X_ml, y_ml, scoring="neg_mean_squared_error", cv=4).mean()

imputers = {"zero": SimpleImputer(strategy="constant", fill_value=0, add_indicator=True),
            "mean": SimpleImputer(strategy="mean", add_indicator=True),
            "KNN (scaled)": make_pipeline(StandardScaler(), KNNImputer(add_indicator=True)),
            "iterative (scaled)": make_pipeline(StandardScaler(), IterativeImputer(random_state=0, add_indicator=True))}
pd.Series({name: cv_mse(imp) for name, imp in imputers.items()}, name="CV MSE of log price").round(4)

zero                  0.2009
mean                  0.2016
KNN (scaled)          0.2034
iterative (scaled)    0.2060
Name: CV MSE of log price, dtype: float64

The differences are small. A random forest can split on the indicator columns and on the filled value itself, so it learns "missing" as its own category; this is why even filling with 0 works. Imputation quality matters more for linear models and distance-based methods. The scikit-learn documentation also notes that some estimators, such as `HistGradientBoostingRegressor`, handle missing values natively and are often a simpler choice than a costly imputation pipeline (Session 10).

> 📖 **Read more:** [extras/04-imputation-methods-compared.ipynb](extras/04-imputation-methods-compared.ipynb) (scikit-learn gallery): the original comparison on the diabetes and California housing data, with values removed on purpose and a bar chart of the errors.

## Worked example · Can we trust the listings? The quality report

*Practice of this block: write a data quality report for the listings (prices stored as text, placeholder values, impossible minimum stays, names in the registration field, duplicates).*

### Step 1 · Collect the checks into one report

Every check in the section on checks wrote one row into `results`. Next we turn them into the report: one row per check, critical findings first. `sort_values(key=...)` sorts the severity by a rank instead of alphabetically.

**Why this step?** Why do the checks that passed stay in the report, with severity "ok"?

In [50]:
rank = {"critical": 0, "major": 1, "minor": 2, "ok": 3}
report = (pd.DataFrame(results)
          .sort_values(["severity", "n_failed"], key=lambda s: s.map(rank) if s.name == "severity" else -s)
          .reset_index(drop=True))
report

,dimension,column,rule,n_failed,severity
0,validity,price (original text),price is a number,6754,critical
1,accuracy,license,registration field holds the name of a private...,1328,major
2,consistency,calendar.listing_id,listing exists in listings (foreign key),79,major
3,consistency,reviews_monthly.listing_id,listing exists in listings (foreign key),57,major
4,accuracy,license,registration field holds the name of a company,1981,minor
5,uniqueness,"host_id, name, room_type, accommodates",same offer as another listing of the host,226,minor
6,consistency,bedrooms,not more bedrooms than guests,31,minor
7,validity,license,registration number in the official format,26,minor
8,validity,minimum_nights,minimum stay at most 365 nights,8,minor
9,validity,maximum_nights,"not the 32-bit placeholder 2,147,483,647",2,minor


### Step 2 · Summarise by dimension and severity

Next we count findings per dimension and severity, the summary line of the report. `unstack` turns the second index level into columns.

**Why this step?** Why count the findings per dimension and severity before deciding on any action?

In [51]:
report.groupby(["dimension", "severity"]).size().unstack(fill_value=0)

severity,critical,major,minor,ok
dimension,,,,
accuracy,0,1,1,0
consistency,0,2,1,3
uniqueness,0,0,1,1
validity,1,0,3,3


### Step 3 · Decide an action for each finding

Add the findings on missing data and imputation (a third of the prices and a quarter of the bedrooms missing, both related to observed columns) and the report answers the practice question. The recommended actions, which block 2 turns into code:

| Finding | Action | Reason |
|---|---|---|
| raw prices are text | parse with the explicit rule; test that all values parse | a rule in code documents itself |
| `maximum_nights` = 2,147,483,647 | set to missing | a software default, not a stay |
| minimum stay above 365 nights | flag, keep | allowed, but not a holiday rental |
| price missing | keep missing, add indicators; never impute for blocked listings | structural |
| review score missing | keep missing, add `no_reviews` | structural |
| bedrooms missing | impute with the guests rule, add an indicator | the rule recovers hidden values well |
| names in the registration field | keep only `license_status` | data minimisation |
| repeated offers of one host | keep; group by host when validating | real, separate units |
| listings in calendar or reviews but not in `listings` | document; use explicit joins | scraped at different times |

## Check your understanding

Vote alone first, discuss with your neighbour, then vote again. Each question has one best answer.

**1.** The check "not more bedrooms than guests" fails for 31 listings. Which dimension of data quality does it belong to?

- **A.** Validity: a number of bedrooms outside its allowed range.
- **B.** Uniqueness: the same flat recorded twice.
- **C.** Consistency: two values of the same row that contradict each other.
- **D.** Completeness: a value that is missing.

**2.** Why does `listings["maximum_nights"].isna().mean()` say almost nothing about the quality of that column?

- **A.** It counts only `NaN`, not placeholders such as 2,147,483,647 or the default 1,125 that mean "no limit".
- **B.** The column is stored as a float, and `isna()` works only on integers.
- **C.** `isna()` ignores zeros, and zeros are the real problem of this column.
- **D.** It returns a share instead of a count.

**3.** A test is marked `@pytest.mark.xfail(strict=True)` because 2 listings carry the placeholder 2,147,483,647. The next snapshot no longer contains them. What does pytest do?

- **A.** It reports the test as passed, and nothing else changes.
- **B.** It skips the test, because the known problem is gone.
- **C.** It never runs the test: `xfail` tests are documentation only.
- **D.** It reports the unexpected pass as a failure, so someone turns the test into a normal test.

**4.** Listings without a recent review lack a price about eight times as often as busy ones (0.589 against 0.069). What explains most of this pattern?

- **A.** Reviews cause prices: without reviews, Airbnb hides the price.
- **B.** A third variable: a listing with no free night shows no price and also collects few reviews, so missingness depends on observed columns (MAR given availability is plausible).
- **C.** Nothing: with 10,000 listings, such differences arise by chance, so the prices are MCAR.
- **D.** The pattern proves that the prices are MNAR.

**5.** A random forest barely cares which imputer filled the gaps. For which model does the choice of imputer matter more?

- **A.** k-nearest neighbours on standardised columns: every filled value moves the distances, and a constant fill puts many rows at the same spot.
- **B.** A larger random forest with more trees.
- **C.** Any model, as long as the test set is imputed separately.
- **D.** None: imputation never changes a model, only the data.

## Exercises

Work in pairs: one types, the other directs. The lecturer picks which exercises to set; there are more than fit into the time in class. The starter cell loads the calendar for exercise 1.5; `listings` is already loaded.

In [52]:
calendar = pd.read_parquet("case-study/data/airbnb/calendar.parquet")
calendar.shape

(3759500, 4)

### 1.1 · Parse ten messy prices

*Code*

> `"$1,200.00"`, `"€80"`, `"80,50"`, `""`, `"N/A"`, `"0"`, `"99999"`, `" 75 "`, `"75.0"`, `"seventy"`. For each, decide: fix, mark as missing or flag. Then write one function that applies your decisions to the Series below and returns numbers.

*Source:* original exercise for this course; background: [`pandas.to_numeric`](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html)

In [53]:
messy = pd.Series(["$1,200.00", "€80", "80,50", "", "N/A", "0", "99999", " 75 ", "75.0", "seventy"])

### 1.2 · Sort the missingness cards

*Class*

> Classify each scenario as MCAR, MAR or MNAR, and say which observed column (if any) explains the missing values.
>
> 1. A temperature sensor fails at random moments.
> 2. Younger respondents skip the income question more often than older ones.
> 3. Hosts of overpriced flats hide the price.
> 4. A survey page did not load for a random 5 % of visitors because of a server error.
> 5. In the Airbnb snapshot, the price is missing for almost every listing that has no free night in the next year.
> 6. Patients in a pain study who are in severe pain drop out before the final pain measurement.

*Source:* [van Buuren, *Flexible Imputation of Missing Data*, §1.2 MCAR, MAR and MNAR](https://stefvanbuuren.name/fimd/sec-MCAR.html); scenarios 1–3 from the course exercise bank, 4–6 original for this course

*Your answer:*

### 1.3 · What does mean imputation do?

*Predict*

> Predict what happens to the variance and to the correlation with another column. Then check your prediction on the entire homes: fill the missing `bedrooms` with the mean and compare the variance of `bedrooms` and its correlation with `accommodates` before and after.

*Source:* [van Buuren, *Flexible Imputation of Missing Data*, §1.3 Simple solutions](https://stefvanbuuren.name/fimd/sec-simplesolutions.html)

*Your answer:*

### 1.4 · The same flat twice

*Code*

> Find likely duplicates: same host, coordinates within about 20 metres, similar title. How many pairs and how many hosts do you find? Are they copies of one row or separate units?

*Source:* original exercise for this course; background: [Record linkage, Wikipedia](https://en.wikipedia.org/wiki/Record_linkage)

### 1.5 · Minimum stays in the calendar

*Code*

> The calendar has a column `minimum_nights` for every listing and date. In how many calendar rows does it differ from the `minimum_nights` of the listing, and for how many listings? Is that a contradiction, or can a host set different minimum stays for different dates? What does it mean for a quality rule on minimum stays?

*Source:* original exercise for this course (Inside Airbnb Berlin case study)

### 1.6 · What date is `03/04/2025`?

*Class*

> Is it 3 April or 4 March? How can the data themselves tell you which format a column uses?

*Source:* original exercise for this course; background: [Date format by country, Wikipedia](https://en.wikipedia.org/wiki/Date_format_by_country)

*Your answer:*

### 1.7 · Are beds and bathrooms missing at random?

*Code*

> *Stretch:* About half of the listings have no value for `beds` and `bathrooms`. Compare the share of missing values by room type, for listings with and without a free night in the next year (`availability_365 == 0`) and for listings with and without a price. Which mechanism is plausible? Would you impute these two columns, and for which listings?

*Source:* original exercise for this course (Inside Airbnb Berlin case study)

### 1.8 · The share of blocked listings without a price

*Find the bug*

> A colleague reports "the share of blocked listings (no free night in the next 365 days) that show no price" with this line:
>
> ```python
> listings[listings["availability_365"].eq(0)]["price"].isna().sum() / len(listings)
> ```
>
> It runs and prints a share of about a quarter. What is wrong, what is the correct share, and how would you write the line so that the mistake cannot happen?

*Source:* original exercise for this course (Inside Airbnb Berlin case study)

*Your answer:*

### 1.9 · Put the test file in order

*Parsons*

> The lines below, with their indentation removed, form a pytest file with one fixture, one normal test and one known problem. Put them in order and indent them. Then predict: which line does pytest print at the end for this file, and with which exit code?
>
> ```python
> def test_room_type_is_known(listings):
> import pytest
> assert listings["minimum_nights"].dropna().le(365).all()
> @pytest.fixture(scope="module")
> return pd.read_parquet(os.environ["LISTINGS"])
> @pytest.mark.xfail(strict=True, reason="known: 8 listings ask for more than 365 nights")
> import os
> def listings():
> assert listings["room_type"].isin(["Entire home/apt", "Private room", "Hotel room", "Shared room"]).all()
> import pandas as pd
> def test_minimum_stay_at_most_one_year(listings):
> ```

*Source:* original exercise for this course

*Your answer:*

## Debrief

- **Compare** two price parsers from exercise 1.1. Look for the one that turns `"80,50"` into 8,050 or treats `"0"` as a valid free night, and ask what a parser should do with values it cannot read: raise, return `NaN`, or flag?
- **Compare** two missingness profiles (1.7). A typical mistake is a claim of MCAR that rests on a single comparison, or imputing beds for listings whose page had no bookable date at all.
- **Duplicates** (1.4): which "duplicates" are separate units of one operator? Uniqueness of an id is not uniqueness of the real flat.
- **Bridge to block 2.** The report lists findings and actions. Block 2 adds the extreme values (prices of several thousand euros a night, minimum stays of 92 nights) and turns every action into a logged step of a cleaning pipeline.

## Further reading

- Kuhn, M., & Johnson, K. (2019). *Feature Engineering and Selection*, chapter 8 "Handling missing data". https://feat.engineering/ (the grouping of this session follows it)
- van Buuren, S. (2018). *Flexible Imputation of Missing Data* (2nd ed.), chapters 1–2. https://stefvanbuuren.name/fimd/
- Rubin, D. B. (1976). Inference and missing data. *Biometrika*, 63(3), 581–592. https://doi.org/10.1093/biomet/63.3.581
- scikit-learn user guide: *Imputation of missing values*. https://scikit-learn.org/stable/modules/impute.html
- pandas user guide: *Working with missing data*. https://pandas.pydata.org/docs/user_guide/missing_data.html
- Schelter, S., et al. (2018). Automating large-scale data quality verification. *PVLDB*, 11(12), 1781–1794. https://doi.org/10.14778/3229863.3229867
- Breck, E., Polyzotis, N., Roy, S., Whang, S. E., & Zinkevich, M. (2019). Data validation for machine learning. *MLSys 2019*.
- Bantilan, N. (2020). pandera: Statistical data validation of pandas dataframes. *Proceedings of the 19th Python in Science Conference*, 116–124. https://pandera.readthedocs.io/
- Inside Airbnb. *Data assumptions*. https://insideairbnb.com/data-assumptions/
- Regulation (EU) 2016/679 (General Data Protection Regulation), Article 5(1)(c): data minimisation. https://eur-lex.europa.eu/eli/reg/2016/679/oj

Curated sources of this notebook: Microsoft, *Data Science for Beginners*, lesson 08 (MIT); pandera, *Try pandera* (MIT); scikit-learn gallery, *Imputing missing values before building an estimator* and *Imputing missing values with variants of IterativeImputer* (BSD-3-Clause); code from VanderPlas, *Python Data Science Handbook* (code MIT). Extras for this block: [01 · missing values and duplicates](extras/01-data-cleaning-missing-and-duplicates.ipynb), [02 · pandera](extras/02-try-pandera.ipynb), [03 · missing values in pandas](extras/03-missing-values-in-pandas.ipynb), [04 · imputation methods compared](extras/04-imputation-methods-compared.ipynb), [05 · iterative imputer variants](extras/05-iterative-imputer-variants.ipynb). Provenance: [extras/README.md](extras/README.md).